Nonimmigrant visa (NIV) issuances by nationality and visa class, February 2021 - February 2026

In [1]:
## parse monthly NIV pdfs from state department
import re, io, zipfile
from pathlib import Path
import pandas as pd
import pdfplumber

# pick the real zip (an earlier failed download left an empty 22-byte zip with the same name)
DL = Path(r"C:\Users\CarolXu\Downloads")
ZIP = max(DL.glob("niv_nationality_pdfs_2021-02_to_2026*.zip"), key=lambda p: p.stat().st_size)
print("Reading", ZIP.name)

# NIV class symbols: D, H1B, H1B1, NATO2, B1/B2, C1/D, BBBCC
row_re  = re.compile(r"^(?P<country>.*?)\s+(?P<cls>[A-Z][A-Z0-9]{0,4}(?:/[A-Z0-9]{1,4})*)\s+(?P<n>\d[\d,]*)$")
gt_re   = re.compile(r"Grand\s+Total\s+([\d,\s]+)$", re.I)   # some PDFs print it as "2 6,483"
page_re = re.compile(r"^Page \d+ of \d+$", re.I)

all_rows, checks, unmatched = [], [], []
with zipfile.ZipFile(ZIP) as z:
    for name in sorted(n for n in z.namelist() if n.endswith(".pdf")):
        date = pd.Timestamp(name[:7] + "-01")          # files are named YYYY-MM_NIV.pdf
        rows, gt = [], None
        with pdfplumber.open(io.BytesIO(z.read(name))) as pdf:
            for page in pdf.pages:
                for line in (page.extract_text() or "").splitlines():
                    line = re.sub(r"\s+", " ", line).strip()
                    if g := gt_re.search(line):
                        gt = int(re.sub(r"\D", "", g.group(1))); continue
                    if m := row_re.match(line):
                        rows.append((date, m["country"].strip(), m["cls"], int(m["n"].replace(",", ""))))
                    elif re.search(r"\d$", line) and not page_re.match(line):
                        unmatched.append((date, line))
        all_rows += rows
        checks.append({"date": date, "parsed_total": sum(r[3] for r in rows), "pdf_grand_total": gt})

detail = pd.DataFrame(all_rows, columns=["date", "country", "visa_class", "issuances"])

# Make country names consistent across months
detail["country"] = (detail.country.str.replace("–", "-", regex=False)
                     .str.strip("* ")
                     .str.replace("Non- Nationality", "Non-Nationality", regex=False)
                     .replace({"China - Taiwan": "Taiwan"}))

checks = pd.DataFrame(checks)
checks["match"] = checks.parsed_total.astype(int) == checks.pdf_grand_total.fillna(-1).astype(int)
print(f"{checks.match.sum()} / {len(checks)} months match the PDF Grand Total")
print(detail.country.nunique(), "countries |", detail.visa_class.nunique(), "visa classes")
print(sorted(detail.visa_class.unique()))
if unmatched:
    print(f"\n{len(unmatched)} unparsed lines ending in a number, e.g.:")
    for d, l in unmatched[:15]:
        print(d.strftime("%Y-%m"), "|", l)

Reading niv_nationality_pdfs_2021-02_to_2026 (1).zip
61 / 61 months match the PDF Grand Total
210 countries | 80 visa classes
['A1', 'A2', 'A3', 'B1', 'B1/B2', 'B2', 'BBBCC', 'BBBCV', 'C1', 'C1/D', 'C2', 'C3', 'C4', 'C4/D3', 'CW1', 'CW2', 'D', 'D3', 'E1', 'E2', 'E2C', 'E3', 'E3D', 'E3R', 'F1', 'F2', 'G1', 'G2', 'G3', 'G4', 'G5', 'H1B', 'H1B1', 'H1C', 'H2A', 'H2B', 'H3', 'H4', 'I', 'J1', 'J2', 'K1', 'K2', 'K3', 'K4', 'L1', 'L2', 'M1', 'M2', 'N8', 'N9', 'NATO1', 'NATO2', 'NATO3', 'NATO4', 'NATO5', 'NATO6', 'NATO7', 'O1', 'O2', 'O3', 'P1', 'P2', 'P3', 'P4', 'Q1', 'R1', 'R2', 'T2', 'T3', 'T4', 'T5', 'T6', 'TD', 'TN', 'U1', 'U2', 'U3', 'U4', 'U5']


In [2]:
## write csvs

detail["year"], detail["month"] = detail.date.dt.year, detail.date.dt.month

# CSV 1: monthly total (ALL COUNTRIES) plus each country's monthly total
by_country = detail.groupby(["date", "year", "month", "country"], as_index=False).issuances.sum()
overall = detail.groupby(["date", "year", "month"], as_index=False).issuances.sum().assign(country="ALL COUNTRIES")
totals = pd.concat([overall, by_country])
totals["_o"] = totals.country.ne("ALL COUNTRIES")
totals = totals.sort_values(["date", "_o", "country"]).drop(columns="_o")
totals["date"] = totals.date.dt.strftime("%Y-%m")
totals[["date", "year", "month", "country", "issuances"]].to_csv("niv_issuances_monthly_totals.csv", index=False)

# CSV 2: by country and visa class
out = detail.sort_values(["date", "country", "visa_class"]).copy()
out["date"] = out.date.dt.strftime("%Y-%m")
out[["date", "year", "month", "country", "visa_class", "issuances"]].to_csv("niv_issuances_by_country_visa_class.csv", index=False)

print(totals.shape, out.shape)

(12405, 5) (210169, 6)
